## Deterministic retrieval on the final development partitions

Search the complete catalog with the same source evidence contract reserved for later Gemma comparisons. Retain products with missing mappings. Predict from input_json alone and read references only during scoring. Exact NDC, formulation and ingredient targets have separate denominators. The sealed test partition receives no predictions.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from hrp.retrieval_baseline import run_evaluation

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
summary = run_evaluation(project_root)
assert summary["test_predictions_generated"] == 0
assert not summary["test_reference_json_decoded"]
assert not summary["test_input_json_decoded"]
print("Catalog NDCs", summary["catalog_ndcs"])
display(pd.DataFrame(summary["catalog_ingredient_support"]).T)


train 81251 cases evaluated


validation 15393 cases evaluated


Catalog NDCs 56300


,numerator,denominator,value
SUPPORTED,53562.0,56300.0,0.951368
UNAVAILABLE,1023.0,56300.0,0.018171
SOURCE_CONFLICT,1715.0,56300.0,0.030462


In [2]:
def fraction(metric):
    value = metric["value"]
    result = f"{metric['numerator']}/{metric['denominator']}"
    return result + (f" ({value:.2%})" if value is not None else " not estimable")

rows = {}
for part in ("train", "validation"):
    result = summary[part]
    product = result["product_resolution"]
    ingredients = product["ingredient_identification"]
    comparison = result["ingredient_comparison"]
    rows[part] = {
        "Cases": result["cases"],
        "Independent partition groups": result["family_groups"],
        "Exact NDC accuracy when justified": fraction(product["exact_ndc_accuracy_when_justified"]),
        "Formulation accuracy with supported references": fraction(product["formulation_accuracy_all_eligible"]),
        "Formulation reference coverage": fraction(product["formulation_reference_coverage"]),
        "Source limited formulation requests": fraction(product["source_limited_formulation_requests"]),
        "Full coverage ingredient set accuracy": fraction(ingredients["set_exact_match_all_eligible"]),
        "Chemical identifier reference coverage": fraction(ingredients["chemical_identifier_reference_coverage"]),
        "Chemical identifier set accuracy": fraction(ingredients["chemical_identifier_set_accuracy_all_covered"]),
        "Source name only set accuracy": fraction(ingredients["source_name_only_set_accuracy"]),
        "Ingredient precision": fraction(ingredients["precision"]),
        "Ingredient recall": fraction(ingredients["recall"]),
        "Ingredient coverage": fraction(ingredients["coverage"]),
        "Unsupported ingredients": fraction(ingredients["unsupported_ingredients_against_reference"]),
        "Correct abstention": fraction(product["correct_abstention"]),
        "Unresolved product matches": fraction(product["unresolved_product_matches"]),
        "No match": fraction(product["no_match"]),
        "Comparison accuracy": fraction(comparison["classification_accuracy_all_cases"]),
        "Overlap precision": fraction(comparison["overlap_precision"]),
        "Overlap recall": fraction(comparison["overlap_recall"]),
        "Shared ingredient precision": fraction(comparison["shared_ingredient_metrics"]["precision"]),
        "Shared ingredient recall": fraction(comparison["shared_ingredient_metrics"]["recall"]),
        "Unresolved comparisons": fraction(comparison["unresolved_comparisons"]),
        "Joint case accuracy": fraction(result["joint_case_accuracy"]),
        "Family macro case accuracy": result["family_macro"]["mean_joint_case_accuracy"],
    }
display(pd.DataFrame(rows))


,train,validation
Cases,81251,15393
Independent partition groups,13935,3013
Exact NDC accuracy when justified,98816/98816 (100.00%),18234/18234 (100.00%)
Formulation accuracy with supported references,43552/43552 (100.00%),7770/7770 (100.00%)
Formulation reference coverage,43552/101742 (42.81%),7770/18834 (41.26%)
Source limited formulation requests,58190/101742 (57.19%),11064/18834 (58.74%)
Full coverage ingredient set accuracy,101742/101742 (100.00%),18834/18834 (100.00%)
Chemical identifier reference coverage,101225/101742 (99.49%),18759/18834 (99.60%)
Chemical identifier set accuracy,101225/101225 (100.00%),18759/18759 (100.00%)
Source name only set accuracy,517/517 (100.00%),75/75 (100.00%)


In [3]:
display(pd.DataFrame({part: summary[part]["cases_by_task"] for part in ("train", "validation")}))
print(json.dumps({part: {"issues": summary[part]["issue_counts"],
                       "source_limited_failures": summary[part]["source_limited_failures"]}
                  for part in ("train", "validation")}, indent=2))
print("Saved aggregate results", "results/final_v5/part04/validation_summary.json")
print("Held-out predictions", summary["test_predictions_generated"])


,train,validation
ambiguity_detection,18656,3826
ingredient_comparison,21421,3631
product_identification,41174,7936


{
  "train": {
    "issues": {},
    "source_limited_failures": {
      "numerator": 0,
      "denominator": 102672,
      "value": 0.0
    }
  },
  "validation": {
    "issues": {},
    "source_limited_failures": {
      "numerator": 0,
      "denominator": 19024,
      "value": 0.0
    }
  }
}
Saved aggregate results results/final_v5/part04/validation_summary.json
Held-out predictions 0


Agreement with these source-supported references measures retrieval and extraction consistency. Both references and retrieval use submitted FDA evidence, so strong scores do not establish independent clinical truth. Unresolved source strength bases remain outside formulation scoring and their counts remain visible. Repeated products and comparisons do not create independent evidence. The original Gemma model and the compact models are compared separately in Notebook 08 using the fixed 230 case challenge. Gemma was not fine tuned, and the original sealed test set remains unused.


FDA chemical identifier sets support chemical overlap. Source name only references measure reported name extraction. Neither scope establishes clinical equivalence. The complete catalog is retrieval evidence and held out answers never enter prediction inputs.